In [1]:
!pip install yfinance

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

In [3]:
data = yf.download(['AAPL', 'MSFT'], start='2015-06-06', end='2025-06-06')
data

C:\Users\91767\AppData\Local\Temp\ipykernel_23628\225989919.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(['AAPL', 'MSFT'], start='2015-06-06', end='2025-06-06')
[*********************100%***********************]  2 of 2 completed


Price            Close                    High                     Low  \
Ticker            AAPL        MSFT        AAPL        MSFT        AAPL   
Date                                                                     
2015-06-08   28.590513   39.586975   28.905949   40.192943   28.373511   
2015-06-09   28.505505   39.517719   28.653156   39.768760   28.102822   
2015-06-10   28.832130   40.348766   28.935036   40.539214   28.601705   
2015-06-11   28.767246   40.201599   29.122949   40.617119   28.742638   
2015-06-12   28.449572   39.794743   28.709079   40.227576   28.436149   
...                ...         ...         ...         ...         ...   
2025-05-30  200.622314  459.604431  201.731057  460.922272  196.556921   
2025-06-02  201.471344  461.211823  201.900864  461.351577  199.893133   
2025-06-03  203.039566  462.210175  203.538999  463.378268  200.732187   
2025-06-04  202.590088  463.108673  206.006209  464.925693  201.870903   
2025-06-05  200.402573  466.912415  204.517897  468.879183  199.923106   

Price                         Open                 Volume            
Ticker            MSFT        AAPL        MSFT       AAPL      MSFT  
Date                                                                 
2015-06-08   39.535034   28.836595   40.080405  210699200  22121600  
2015-06-09   39.353240   28.344431   39.612940  224301600  24406100  
2015-06-10   39.552350   28.617364   39.638919  156349200  28417400  
2015-06-11   39.933244   28.899236   40.392047  141563600  27347800  
2015-06-12   39.734146   28.677760   40.011159  147544800  23931000  
...                ...         ...         ...        ...       ...  
2025-05-30  454.792365  199.143981  458.965497   70819900  34770500  
2025-06-02  456.140173  200.052956  456.389763   35423300  16626500  
2025-06-03  460.103622  201.121744  460.712636   46381600  15743800  
2025-06-04  462.260062  202.679982  463.238465   43604000  14162700  
2025-06-05  463.268411  203.269314  464.196878   55126100  20131700  

[2515 rows x 10 columns]

In [8]:
data.to_csv('AAPL_stock_data.csv')

In [9]:
import os

os.getcwd()

'C:\\Users\\91767\\yfinance'

In [11]:
data.to_csv('MSFT_stock_data.csv')


<function nt.getcwd()>

In [12]:
import os
os.getcwd()

'C:\\Users\\91767\\yfinance'

In [5]:
data.isnull().sum()

Price   Ticker
Close   AAPL      0
        MSFT      0
High    AAPL      0
        MSFT      0
Low     AAPL      0
        MSFT      0
Open    AAPL      0
        MSFT      0
Volume  AAPL      0
        MSFT      0
dtype: int64

In [8]:
def infer_sql_types(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return 'INT'
    elif pd.api.types.is_bool_dtype(dtype):
        return 'BOOLEAN'
    elif pd.api.types.is_float_dtype(dtype):
        return 'FLOAT'
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return 'TIMESTAMP'
    else:
        return 'TEXT'
table_name = 'Stocks_data'
columns = data.dtypes
sql_columns = ',\n '.join([f'"{col}"{infer_sql_types(dtype)}' for col, dtype in columns.items()])

create_stmt = f"""
CREATE TABLE IF NOT EXISTS {table_name} (
 {sql_columns}
);
"""

print(create_stmt)


CREATE TABLE IF NOT EXISTS Stocks_data (
 "('Close', 'AAPL')"FLOAT,
 "('Close', 'MSFT')"FLOAT,
 "('High', 'AAPL')"FLOAT,
 "('High', 'MSFT')"FLOAT,
 "('Low', 'AAPL')"FLOAT,
 "('Low', 'MSFT')"FLOAT,
 "('Open', 'AAPL')"FLOAT,
 "('Open', 'MSFT')"FLOAT,
 "('Volume', 'AAPL')"INT,
 "('Volume', 'MSFT')"INT
);



In [9]:
import psycopg2
conn = psycopg2.connect(
    dbname = "postgres",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)

cur = conn.cursor()
# Create table manually here or auto-generate
cur.execute(create_stmt)
conn.commit()

# Auto-generate INSERT statement
columns = list(data.columns)
placeholders = ', '.join(['%s'] * len(columns))
insert_stmt = f"""INSERT INTO {table_name} ({', '.join(['"{}"'.format(col) for col in columns])}) 
VALUES ({placeholders})
"""

# Insert data row by row
for _, row in data.iterrows():
    cur.execute(insert_stmt, tuple(row))


conn.commit()
cur.close()
conn.close()